# Deep Learning: Brain Decoding with PyTorch

Train a CNN to decode visual categories from fMRI BOLD patterns (Haxby dataset). Compare linear SVM vs deep network. This is the core idea behind modern brain-computer interfaces and neural decoding.

**Frontier context:** This builds toward models like MindEye2 (2024) which reconstructs seen images from fMRI using diffusion model priors.

## Prerequisites

- **Python 3.8+** with PyTorch installed
- Familiarity with basic deep learning concepts (MLPs, backpropagation, cross-entropy loss)
- Understanding of fMRI BOLD signal and MVPA (multi-voxel pattern analysis)
- The Haxby dataset will be downloaded automatically by nilearn (~300 MB)

## Setup

Run the cell below to install all required packages. PyTorch will use CUDA if available, otherwise CPU.

In [ ]:
# Install required packages
!pip install -q torch torchvision numpy matplotlib scikit-learn nilearn pandas

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import nilearn.datasets as nld
from nilearn.maskers import NiftiMasker

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
# Load Haxby dataset -- classic MVPA benchmark
haxby = nld.fetch_haxby()
import pandas as pd
labels = pd.read_csv(haxby.session_target[0], sep=' ')

# Use only 6 visual categories (exclude 'rest')
categories = ['face', 'house', 'cat', 'bottle', 'scissors', 'shoe']
mask = labels['labels'].isin(categories)
y = labels['labels'][mask].values
print('Categories:', np.unique(y))
print('Samples per class:', {c: (y==c).sum() for c in np.unique(y)})

# Extract BOLD patterns using ventral temporal mask
nifti_masker = NiftiMasker(mask_img=haxby.mask_vt[0],
                            standardize=True, memory_level=1)
X = nifti_masker.fit_transform(haxby.func[0])[mask]
print(f'\nFeature matrix: {X.shape}  (trials x voxels)')

In [ ]:
# Baseline: Linear SVM (the classic MVPA approach)
svm = make_pipeline(StandardScaler(), SVC(kernel='linear', C=1.0))
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_enc = le.fit_transform(y)
svm_scores = cross_val_score(svm, X, y_enc, cv=5, scoring='accuracy')
print(f'Linear SVM: {svm_scores.mean():.3f} +/- {svm_scores.std():.3f} (chance = {1/len(categories):.2f})')

In [ ]:
# Deep network: small MLP for brain decoding
class BrainDecoder(nn.Module):
    def __init__(self, n_voxels, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_voxels, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, 256),      nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, 128),      nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, n_classes)
        )
    def forward(self, x): return self.net(x)

# Train/test split
from sklearn.model_selection import train_test_split
scaler = StandardScaler()
X_s = scaler.fit_transform(X)
X_tr, X_te, y_tr, y_te = train_test_split(X_s, y_enc, test_size=0.2, random_state=42, stratify=y_enc)

X_tr_t = torch.FloatTensor(X_tr).to(device)
X_te_t = torch.FloatTensor(X_te).to(device)
y_tr_t = torch.LongTensor(y_tr).to(device)
y_te_t = torch.LongTensor(y_te).to(device)

model = BrainDecoder(X.shape[1], len(categories)).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

# Training loop
train_losses, train_accs, val_accs = [], [], []
for epoch in range(80):
    model.train()
    optimizer.zero_grad()
    out = model(X_tr_t)
    loss = criterion(out, y_tr_t)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())
    train_accs.append((out.argmax(1) == y_tr_t).float().mean().item())
    model.eval()
    with torch.no_grad():
        val_pred = model(X_te_t).argmax(1)
        val_accs.append((val_pred == y_te_t).float().mean().item())
    if (epoch+1) % 20 == 0:
        print(f'Epoch {epoch+1:3d} | Loss: {loss.item():.3f} | Train acc: {train_accs[-1]:.3f} | Val acc: {val_accs[-1]:.3f}')

print(f'\nFinal -- MLP: {val_accs[-1]:.3f} vs SVM: {svm_scores.mean():.3f}  (chance: {1/len(categories):.2f})')

In [ ]:
# Plot learning curves + confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(train_accs, label='Train accuracy')
ax1.plot(val_accs, label='Val accuracy')
ax1.axhline(1/len(categories), color='gray', linestyle='--', label='Chance')
ax1.axhline(svm_scores.mean(), color='orange', linestyle='--', label='SVM baseline')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.set_title('Brain Decoder Training'); ax1.legend()

model.eval()
with torch.no_grad():
    preds = model(X_te_t).argmax(1).cpu().numpy()
# Use string labels for both axes -- convert y_te (numeric) back to class names
y_te_labels = le.inverse_transform(y_te)
y_pred_labels = le.inverse_transform(preds)
cm = confusion_matrix(y_te_labels, y_pred_labels, labels=le.classes_)
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=ax2, colorbar=False)
ax2.set_title('Confusion Matrix -- Visual Category Decoding')
plt.tight_layout(); plt.show()

## References

- Haxby, J. V., Gobbini, M. I., Furey, M. L., et al. (2001). Distributed and overlapping representations of faces and objects in ventral temporal cortex. *Science*, 293(5539), 2425-2430. https://doi.org/10.1126/science.1063736
- Norman, K. A., Polyn, S. M., Detre, G. J., & Haxby, J. V. (2006). Beyond mind-reading: multi-voxel pattern analysis of fMRI data. *Trends in Cognitive Sciences*, 10(9), 424-430.
- Scotti, P. S., Banerjee, A., Gober, J., et al. (2024). MindEye2: Shared-subject models enable fMRI-to-image with 1 hour of data. *Proc. ICML*. https://arxiv.org/abs/2403.11207
- Paszke, A., Gross, S., Massa, F., et al. (2019). PyTorch: An imperative style, high-performance deep learning library. *NeurIPS*, 32.